# Stitching together CANDOR files from the checkpoints

Perhaps inadvertently, I've saved the outputs for each conversation as a CEDA checkpoint. We need to iterate through them and stitch together the final file for analysis.

In [1]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

In [2]:
DATA_PATH = 'data'
CKPT_PATH = os.path.join(DATA_PATH, 'ckpts')

OUTPUT_PATH = os.path.join(DATA_PATH, 'lme_ready')
if not os.path.exists(OUTPUT_PATH):
    os.mkdir(OUTPUT_PATH)

OUTPUT_NAME = os.path.join(OUTPUT_PATH,'candor-ceda-analysis.tsv')

In [3]:
def grab_all_files(PATH, file_type='.cha'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

## Grabbing each file and creating a merged dataframe

In [4]:
files = grab_all_files(CKPT_PATH, '.pt')
print(files)

['data/ckpts/graph-obj-2.csv.pt', 'data/ckpts/graph-obj-31.csv.pt', 'data/ckpts/graph-obj-23.csv.pt', 'data/ckpts/graph-obj-15.csv.pt', 'data/ckpts/graph-obj-19.csv.pt', 'data/ckpts/graph-obj-21.csv.pt', 'data/ckpts/graph-obj-33.csv.pt', 'data/ckpts/graph-obj-17.csv.pt', 'data/ckpts/graph-obj-0.csv.pt', 'data/ckpts/graph-obj-13.csv.pt', 'data/ckpts/graph-obj-29.csv.pt', 'data/ckpts/graph-obj-25.csv.pt', 'data/ckpts/graph-obj-8.csv.pt', 'data/ckpts/graph-obj-4.csv.pt', 'data/ckpts/graph-obj-6.csv.pt', 'data/ckpts/graph-obj-11.csv.pt', 'data/ckpts/graph-obj-27.csv.pt', 'data/ckpts/graph-obj-14.csv.pt', 'data/ckpts/graph-obj-18.csv.pt', 'data/ckpts/graph-obj-30.csv.pt', 'data/ckpts/graph-obj-22.csv.pt', 'data/ckpts/graph-obj-3.csv.pt', 'data/ckpts/graph-obj-1.csv.pt', 'data/ckpts/graph-obj-16.csv.pt', 'data/ckpts/graph-obj-20.csv.pt', 'data/ckpts/graph-obj-32.csv.pt', 'data/ckpts/graph-obj-9.csv.pt', 'data/ckpts/graph-obj-5.csv.pt', 'data/ckpts/graph-obj-28.csv.pt', 'data/ckpts/graph-obj-

In [5]:
df_ = []
for f in tqdm(files):
    ckpt = torch.load(f)
    df = pd.DataFrame(ckpt['meta_data'])
    
    # number of tokens per text
    df['nx'] = ckpt['N'][:,0].tolist()
    df['ny'] = ckpt['N'][:,1].tolist()
    
    # CE values
    df['Hxy'] = ckpt['M'][:,0].tolist()
    df['Hyx'] = ckpt['M'][:,1].tolist()
    
    df_ += [df]

df = pd.concat(df_, ignore_index=True)
print(df.shape)
df.head()

  0%|          | 0/34 [00:00<?, ?it/s]

(429480, 51)


,x_turn_id,x_speaker,x_start,x_stop,x_delta,x_race,x_politics,x_age,x_sex,x_edu,...,y_i_think_my_status,y_i_think_your_status,combined_race,combined_sex,combined_edu,conversation_id,nx,ny,Hxy,Hyx
0,0,5f13bae4a544193e6b94b0cf,13.01,212.54,199.53,white,3.0,38.0,female,some_college,...,9.0,8.0,asian-white,female-female,some_college-some_college,candor-2.csv,131.0,7.0,17.017906,1.345079e+00
1,0,5f13bae4a544193e6b94b0cf,13.01,212.54,199.53,white,3.0,38.0,female,some_college,...,9.0,8.0,asian-white,female-female,some_college-some_college,candor-2.csv,131.0,4.0,19.524956,5.936596e-01
2,0,5f13bae4a544193e6b94b0cf,13.01,212.54,199.53,white,3.0,38.0,female,some_college,...,9.0,8.0,asian-white,female-female,some_college-some_college,candor-2.csv,131.0,13.0,13.156568,1.659376e+00
3,0,5f13bae4a544193e6b94b0cf,13.01,212.54,199.53,white,3.0,38.0,female,some_college,...,9.0,8.0,asian-white,female-female,some_college-some_college,candor-2.csv,131.0,14.0,20.172615,3.210254e+00
4,0,5f13bae4a544193e6b94b0cf,13.01,212.54,199.53,white,3.0,38.0,female,some_college,...,9.0,8.0,asian-white,female-female,some_college-some_college,candor-2.csv,131.0,3.0,17.420267,1.192093e-07


#### Creating Add'l Regression Model Variables

In [6]:
# df['age_dif'] = df['x_age'] - df['y_age']
# df['pol_dif'] = df['x_politics'] - df['y_politics']
# df['status_dif'] = df['x_i_think_my_status'] - df['x_i_think_your_status']
# df['same_gender'] = (df['x_sex'] == df['y_sex']).astype(int)
# df['same_race'] = (df['x_race'] == df['y_race']).astype(int)
# df['same_edu'] = (df['x_edu'] == df['y_edu']).astype(int)
# df['t_delta'] = df['y_turn_id'] - df['x_turn_id']

In [7]:
# df[['age_dif', 'pol_dif', 'status_dif', 'same_gender', 'same_race', 't_delta', 'same_edu']].mean()

In [8]:
# df[['age_dif', 'pol_dif', 'status_dif', 'same_gender', 'same_race', 't_delta', 'same_edu']].median()

In [9]:
# df[['age_dif', 'pol_dif', 'status_dif', 'same_gender', 'same_race', 't_delta', 'same_edu']].std()

## Save data

In [10]:
df.to_csv(
    OUTPUT_NAME,
    index=False, 
    encoding='utf-8',
    sep='\t'
)